### Note: Please ensure that this notebook is run in the daq environment

In [1]:
import glob
import os
import tqdm
from collections import defaultdict
import re
import numpy as np
import pandas as pd
from scipy.io import wavfile
from moviepy.video.io.ffmpeg_tools import ffmpeg_extract_subclip
from moviepy.video.io.VideoFileClip import VideoFileClip

In [18]:
experiment_no = 105
no_channels = 14
camera_fps = 30
trig_channel = 13 # neural trigger channel
sr = 125000
file_length = 360 # in seconds (360 sec - 6 mins)
# camera_corrected_sample_time = 0.03338542
nidaq_folder = "D:/big_setup/experiment_{}/nidaq/".format(experiment_no)
video_folder = "D:/big_setup/experiment_{}/videos/".format(experiment_no)
neural_folder = "D:/big_setup/experiment_{}/neural/".format(experiment_no)
cam_mic_sync_concat_folder = "D:/big_setup/experiment_{}/concatenated_data_cam_mic_sync/".format(experiment_no)
timestamp_file = "D:/big_setup/experiment_{}/camera_timestamps.csv".format(experiment_no)

In [19]:
# Deleting the trunc files if it is already present
trunc_files = glob.glob(nidaq_folder+"*_trunc*")
for i in trunc_files:
    os.remove(i) 

trunc_files = glob.glob(video_folder+"*_trunc*")
for i in trunc_files:
    os.remove(i) 

In [20]:
nidaq_files = glob.glob(nidaq_folder+"*")
video_files = glob.glob(video_folder+"*")

In [49]:
# Creating separate folders for the concatenated data
path_concat = "D:/big_setup/experiment_{}/concatenated_data_cam_mic_neural_sync/".format(experiment_no)

try:
    os.makedirs(path_concat)
except:
    pass

In [23]:
# Functions to sort file names correctly
def atoi(text):
    return int(text) if text.isdigit() else text

def natural_keys(text):
    return [ atoi(c) for c in re.split(r'(\d+)', text) ]

In [24]:
nidaq_files.sort(key=natural_keys)
video_files.sort(key=natural_keys)

In [25]:
nidaq_files = [i.replace("\\",'/') for i in nidaq_files]
video_files = [i.replace("\\",'/') for i in video_files]

### Defining functions we need

In [26]:
# Function to load the neural data binary file
def load_wm(bin_file):
	"""
	Load .bin file from white matter electophysiology system.

	Parameters:
	filenme: str
		Path to .bin file

	Returns:
	sr: int
		Sampling rate of physiology experiment.
	data: 2-D array
		(n_channels, n_samples) numpy array.
	"""

	#parse filename to get number of channels 
	n_channels = int(bin_file.split('_')[-2][:-2])

	print('Loading data')
	#load in binary data
	_data = np.fromfile(bin_file,'int16', offset=8)

	#reshape data to (n_samples, n_samples) and scale values to MICROVOLTS
	data = _data.reshape(-1,n_channels)*6.25e3/32768
	data = data.T.astype('int16') #transpose and convert back to int 16
	
	#parse filename to get sampling rate
	sr = int(bin_file.split('_')[-1][:-7])

	
	return sr, data



In [27]:
def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx],idx

### Finding the trigger

In [28]:
# Getting all the trigger channels
trig_channels = glob.glob(nidaq_folder+f"*_{trig_channel}.wav")
trig_channels.sort(key=natural_keys)

In [29]:
no_of_triggers_found = 0 # This is used to keep track of the number of triggers

# Storing the indices of the rising edges in a list
indices_r = []

print("Getting the rising edges in the neural trig signal")
for reads,channel in enumerate(trig_channels):
    sampling_rate,ch = wavfile.read(channel)
    
    # Getting all the rising and falling edges in the neural trig channels
    threshold_r_low = 0.9 # To detect all rising edges within the threshold
    threshold_r_high = 9

    temp_r = []

    diff = ch[1:]-ch[0:-1]
    temp_r = np.where((diff>threshold_r_low) & (diff < threshold_r_high))[0] # The diff < 9 is to prevent detecting the stop nidaq record edge which reads a value 10


    if len(temp_r)>0:
        temp_r = temp_r.astype(np.int64)
        temp_r+=len(ch)*reads
        if len(temp_r>1):
            # Removing multiple detections of the same rising edge
            # trig_edge_freq = 10 # This is used to estimate when the next rising edge can be detected
            # window_samples = int((1/camera_sampling_rate)*sampling_rate) # Number of samples present in the window
            window_samples = 0.1 * sr

            indices_r = np.array([],dtype=np.int64)
            prev_sample = temp_r[0]
            indices_r = np.append(indices_r,prev_sample)
            for i in temp_r[1:]:
                if not(abs(i - prev_sample) < window_samples):
                    indices_r = np.append(indices_r,i)
                prev_sample = i
        else:
            indices_r.append(temp_r[0])
        
        no_of_triggers_found += len(indices_r)   

Getting the rising edges in the neural trig signal


In [30]:
# Finding the start point of camera recording from the timestamp file
timestamp = pd.read_csv(timestamp_file)

In [31]:
# Getting the first camera sample recording sample index
rec_start = timestamp["concat_clk_ch_sample_idx"][0]

In [32]:
indices_r

array([7455999], dtype=int64)

In [33]:
rec_start

3816193

### Formatting and storing the neural data

In [35]:
# Checking for the neural file names
neural_chs_names = glob.glob(neural_folder + "*.bin")
neural_chs_names.sort(key=natural_keys)

In [36]:
if len(neural_chs_names) != len(indices_r):
    raise ValueError("The number of neural recording starts don't match the number of files")

In [50]:
# storing the lengths of the neural data files
lengths_neural = []
neural_data_remove = []
# Reading the neural data to get the lengths
for idx,name in enumerate(neural_chs_names):
    sr_neural,neural_data = load_wm(name)

    neural_len = np.float64(len(neural_data[0]))/np.float64(sr_neural)

    # Checking if the neural data recording was started before the camera recording
    if rec_start >= indices_r[idx]:
        neural_data_remove.append(rec_start-indices_r[idx])
        lengths_neural.append((neural_len*sr-neural_data_remove[-1])/sr)
    else:
        neural_data_remove.append(0)
        lengths_neural.append(neural_len)
    
    try:
        os.makedirs(path_concat+f"recording_{idx}/")
    except:
        pass
    
    for idx_,channel in enumerate(neural_data):
        print(f"Saving neural channel {idx_} of recording {idx}")
        start_sec = neural_data_remove[-1]/sr
        file_breaks  = np.arange(start_sec,lengths_neural[-1],file_length)   
        for ind,breaks in enumerate(file_breaks):
            wavfile.write(path_concat+f"recording_{idx}/neural_channel_{idx_}_{ind}.wav", sr_neural, channel[int(breaks*sr_neural):int(breaks*sr_neural)+int(file_length*sr_neural)])

Loading data
Saving neural channel 0 of recording 0


### For Videos

In [56]:
# Getting the camera names
temp_cam_name = []
for i in video_files:
    temp_cam_name.append(i.split("/")[-1].split("-")[0])
camera_names = list(set(temp_cam_name))
print(camera_names)

['right', 'front', 'back', 'center', 'left']


In [57]:
# Creating a dictionary of all videos of a single camera
video_by_camera = defaultdict(lambda:[])
for idx,name in enumerate(temp_cam_name):
    video_by_camera[name].append(video_files[idx])

In [58]:
cam_clk_data = pd.read_csv(f"D:/big_setup/experiment_{experiment_no}/camera_timestamps.csv")

In [ ]:
for idx,length in enumerate(lengths_neural):
        if rec_start >= indices_r[idx]:
            crop_start = 0.0
            crop_end = length
        else:
            crop_start = (indices_r[idx] - rec_start)/sr
            crop_end = crop_start + length


        idx_start = crop_start

        

Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful
Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful
Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful
Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful
Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful


  0%|          | 0/14 [00:00<?, ?it/s]ffmpeg version 4.3.1 Copyright (c) 2000-2020 the FFmpeg developers
  built with gcc 10.2.1 (GCC) 20200726
  configuration: --disable-static --enable-shared --enable-gpl --enable-version3 --enable-sdl2 --enable-fontconfig --enable-gnutls --enable-iconv --enable-libass --enable-libdav1d --enable-libbluray --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libopus --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libsrt --enable-libtheora --enable-libtwolame --enable-libvpx --enable-libwavpack --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libzimg --enable-lzma --enable-zlib --enable-gmp --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvo-amrwbenc --enable-libmysofa --enable-libspeex --enable-libxvid --enable-libaom --enable-libgsm --enable-librav1e --disable-w32threads --enable-libmfx --enable-ffnvcodec --enable-cuda

In [13]:
file_breaks  = np.arange(0,cam_clk_data.tail(1).index[0]/camera_fps,file_length)
timestamp_idx = []
for val in file_breaks:
    timestamp_idx.append((np.abs(cam_clk_data[f'concat_{camera_names[0]}_time_from_vid_start'] - val)).argmin())

In [14]:

for cam in camera_names:
    file_break_idx = 1
    file_concat_index = 0 
    filename_txt = path_concat + f"video_{cam}_{file_concat_index}.txt"
    f = open(filename_txt, 'w')
    no_videos = len(video_by_camera[cam])

    for index, vid_name in enumerate(video_by_camera[cam]):


        try:
            temp = cam_clk_data.iloc[timestamp_idx[file_break_idx]][f'{cam}_file_name'].split('\\')
        except:
            pass
        
        if temp[0]+'/'+temp[1] == vid_name and index != no_videos -1 :
            break_time = cam_clk_data.iloc[timestamp_idx[file_break_idx]][f'{cam}_time_from_vid_start']
            target_file = temp[0]+"/"+temp[1].split(".")[0]+"_trunc_0.mp4"
            ffmpeg_extract_subclip(vid_name, 0.0, break_time, targetname=target_file)
            f.write("file \'{}\'\n".format(target_file))
            f.close()


            target_file = temp[0]+"/"+temp[1].split(".")[0]+"_trunc_1.mp4"
            with VideoFileClip(vid_name) as video:
                end_time = video.duration
            ffmpeg_extract_subclip(vid_name, break_time, end_time, targetname=target_file)
            file_concat_index+=1
            file_break_idx+=1
            filename_txt = path_concat + f"video_{cam}_{file_concat_index}.txt"
            f = open(filename_txt, 'w')
            f.write("file \'{}\'\n".format(target_file))


        elif temp[0]+'/'+temp[1] != vid_name and index == no_videos -1:
            cam_stop_rec_video_name = cam_clk_data[f"{cam}_file_name"].iloc[-1]
            cam_stop_rec_frame = int(cam_clk_data[f"{cam}_frame_idx"].iloc[-1])
            
            target_file = vid_name.split(".")[0]+"_trunc_0.mp4"
            ffmpeg_extract_subclip(cam_stop_rec_video_name, 0.0, (cam_stop_rec_frame+1)/camera_fps, targetname=target_file)

            f.write("file \'{}\'\n".format(target_file))
            f.close()

        elif temp[0]+'/'+temp[1] == vid_name and index == no_videos -1:
            cam_stop_rec_video_name = cam_clk_data[f"{cam}_file_name"].iloc[-1]
            cam_stop_rec_frame = int(cam_clk_data[f"{cam}_frame_idx"].iloc[-1])

            break_time = cam_clk_data.iloc[timestamp_idx[1]][f'{cam}_time_from_vid_start']
            target_file = temp[0]+"/"+temp[1].split(".")[0]+"_trunc_0.mp4"
            ffmpeg_extract_subclip(vid_name, 0.0, break_time, targetname=target_file)
            f.write("file \'{}\'\n".format(target_file))
            f.close()

            
            target_file = temp[0]+"/"+temp[1].split(".")[0]+"_trunc_1.mp4"
            end_time = (cam_stop_rec_frame+1)/camera_fps
            ffmpeg_extract_subclip(vid_name, break_time, end_time, targetname=target_file)
            file_concat_index+=1
            filename_txt = path_concat + f"video_{cam}_{file_concat_index}.txt"
            f = open(filename_txt, 'w')
            f.write("file \'{}\'\n".format(target_file))
            f.close()

        elif index != no_videos -1:
            f.write("file \'{}\'\n".format(vid_name))

        

Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful
Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful
Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful
Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful
Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful
Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful
Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful
Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful
Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful
Moviepy - Running:
>>> "+ " ".join(cmd)
Moviepy - Command successful


In [86]:
# # Reading the timestamps data for the camera clock channel 
# cam_clk_data = pd.read_csv(f"D:/big_setup/experiment_{experiment_no}/camera_timestamps.csv")
# for cam in camera_names:
#     no_videos = len(video_by_camera[cam])
#     filename_txt = path_concat + "video_{}.txt".format(cam)
#     f = open(filename_txt, 'w')
#     for index, vid_name in enumerate(video_by_camera[cam]):
#         if index != no_videos -1 : 
#             f.write("file \'{}\'\n".format(vid_name))
#         else: # truncating the last few seconds of the last video clip based on the timestamps generated
#             cam_stop_rec_video_name = cam_clk_data[f"{cam}_file_name"].iloc[-1]
#             cam_stop_rec_frame = int(cam_clk_data[f"{cam}_frame_idx"].iloc[-1])
#             print(cam_stop_rec_frame)
#             print(cam_stop_rec_video_name)
#             ffmpeg_extract_subclip(cam_stop_rec_video_name, 0.0, (cam_stop_rec_frame+1)/camera_fps, targetname=cam_stop_rec_video_name.split("\\")[0]+"/"+cam_stop_rec_video_name.split("\\")[1].split(".")[0]+"_trunc.mp4")
#             f.write("file \'{}\'\n".format(cam_stop_rec_video_name.split("\\")[0]+"/"+cam_stop_rec_video_name.split("\\")[1].split(".")[0]+"_trunc.mp4"))
 
#     f.close()


In [87]:
# Used for testing
# cam_stop_rec_video_name.split("\\")[0]+"/"+cam_stop_rec_video_name.split("\\")[1].split(".")[0]+"_trunc.mp4"

In [88]:
# for item in tqdm.tqdm(video_by_camera.items()):
#     fps = 30
#     resolution = (1600,1200) 
    
#     filename_txt = path_concat + "video_{}.txt".format(item[0])
#     f = open(filename_txt, 'w')

#     for idx, vid in enumerate(item[1]):
#             # if idx != (len(item[1])-1):
#             f.write("file \'{}\'\n".format(vid))
#     f.close()
    

### For NIDAQ

In [15]:
# Reading the timestamps data for the camera clock channel 
audio_clk_data = pd.read_csv(f"D:/big_setup/experiment_{experiment_no}/camera_timestamps.csv")
audio_start_rec_idx = int(cam_clk_data["clk_ch_file_name"][0].split("\\")[1].split("_")[2])
audio_stop_rec_idx = int(cam_clk_data["clk_ch_file_name"].iloc[-1].split("\\")[1].split("_")[2])
start_sample_index =  int(cam_clk_data["clk_ch_sample_idx"][0])
end_sample_index =  int(cam_clk_data["clk_ch_sample_idx"].iloc[-1])

In [16]:
# Concatenating the microphone channels
for i in range(no_channels):
    file_break_idx = 1
    file_concat_index = 0 # index for the text files used to concatenate data
    flag = 0 # to indicate camera start record index has reached
    filename_txt = path_concat + f"channel_{i}_{file_concat_index}.txt"
    f = open(filename_txt, 'w')
    for idx,j in tqdm.tqdm(enumerate(nidaq_files[i::no_channels])):
            
            try:
                break_channel_idx = int(audio_clk_data.iloc[timestamp_idx[file_break_idx]][f'clk_ch_file_name'].split('_')[-2])
            except:
                 pass

            if idx == audio_start_rec_idx:            
                # Loading the file
                sampling_rate,ch = wavfile.read(j)
                ch = ch[start_sample_index:]
                target_file = j.split(".")[0]+"_trunc_0.wav"
                wavfile.write(target_file,sampling_rate,ch)
                flag = 1
                f.write("file \'{}\'\n".format(target_file))

            elif idx == break_channel_idx and idx!= audio_stop_rec_idx:

                break_sample_idx = audio_clk_data.iloc[timestamp_idx[file_break_idx]][f'clk_ch_sample_idx']
    
                # Loading the file
                sampling_rate,ch = wavfile.read(j)
                ch = ch[:break_sample_idx]
                target_file = j.split(".")[0]+"_trunc_0.wav"
                wavfile.write(target_file,sampling_rate,ch)

                f.write("file \'{}\'\n".format(target_file))
                f.close()

                
                target_file = j.split(".")[0]+"_trunc_1.wav"
                # Loading the file
                sampling_rate,ch = wavfile.read(j)
                ch = ch[break_sample_idx:]
                wavfile.write(target_file,sampling_rate,ch)

                file_concat_index+=1
                file_break_idx+=1
                filename_txt = path_concat + f"channel_{i}_{file_concat_index}.txt"
                f = open(filename_txt, 'w')
                f.write("file \'{}\'\n".format(target_file))

            elif idx != break_channel_idx and idx == audio_stop_rec_idx:
                    
            
                target_file = j.split(".")[0]+"_trunc_0.wav"

                # Loading the file
                sampling_rate,ch = wavfile.read(j)
                ch = ch[:end_sample_index]
                wavfile.write(target_file,sampling_rate,ch)
                f.write("file \'{}\'\n".format(target_file))
                f.close()

            elif idx == break_channel_idx and idx == audio_stop_rec_idx:

                break_sample_idx = audio_clk_data.iloc[timestamp_idx[file_break_idx]][f'clk_ch_sample_idx']


                target_file = j.split(".")[0]+"_trunc_0.wav"
                # Loading the file
                sampling_rate,ch = wavfile.read(j)
                ch = ch[:break_sample_idx]
                wavfile.write(target_file,sampling_rate,ch)
                f.write("file \'{}\'\n".format(target_file))
                f.close()

            
                target_file = j.split(".")[0]+"_trunc_1.wav"
                # Loading the file
                sampling_rate,ch = wavfile.read(j)
                ch = ch[break_sample_idx:end_sample_index]
                wavfile.write(target_file,sampling_rate,ch)

                file_concat_index+=1
                filename_txt = path_concat + f"channel_{i}_{file_concat_index}.txt"
                f = open(filename_txt, 'w')
                f.write("file \'{}\'\n".format(target_file))
                f.close()

            else:
                f.write("file \'{}\'\n".format(j))

3it [00:15,  5.30s/it]
3it [00:16,  5.43s/it]
3it [00:15,  5.18s/it]
3it [00:15,  5.21s/it]
3it [00:15,  5.14s/it]
3it [00:15,  5.16s/it]
3it [00:15,  5.19s/it]
3it [00:15,  5.25s/it]
3it [00:15,  5.05s/it]
3it [00:14,  4.97s/it]
3it [00:15,  5.03s/it]
3it [00:15,  5.03s/it]
3it [00:15,  5.10s/it]
3it [00:15,  5.01s/it]


In [17]:
# # Concatenating the microphone channels
# flag = 0 # to indicate camera start record index has reached
# for i in range(no_channels):
#     concat_mic_channel = np.array([])
#     sampling_rate = 192000
#     filename_wav = path_concat + "channel_{}.txt".format(i)
#     f = open(filename_wav, 'w')
    
#     for idx,j in tqdm.tqdm(enumerate(nidaq_files[i::no_channels])):
#         if idx == cam_start_rec_idx:
#             # Loading the file
#             sampling_rate,ch = wavfile.read(j)
#             ch = ch[start_sample_index:]
#             wavfile.write(j.split(".")[0]+"_trunc.wav",sampling_rate,ch)
#             j = j.split(".")[0]+"_trunc.wav"
#             # file_lengths+=(300-)
#             flag = 1 
#         elif idx == cam_stop_rec_idx:
#             # Loading the file
#             sampling_rate,ch = wavfile.read(j)
#             ch = ch[:end_sample_index]
#             wavfile.write(j.split(".")[0]+"_trunc.wav",sampling_rate,ch)
#             j = j.split(".")[0]+"_trunc.wav"
#             flag = 2
#         if flag == 0:
#             continue
#         f.write("file \'{}\'\n".format(j))
#         if flag == 2:
#             break
    
#     f.close()


## Steps done via code in the below cell 
- open anaconda prompt
- Enter the following commands:
    - For video concatenation:
        - ```conda activate daq```
        - ```ffmpeg -f concat -safe 0 -i {path_to_text_file} -c copy {path_to_output_file}```
            
            eg: ```ffmpeg -f concat -safe 0 -i D:\big_setup\experiment_9\concatenated_data_cam_mic_sync\video_e3v83b3.txt -c copy D:\big_setup\experiment_9\concatenated_data_cam_mic_sync\e3v83b3_ffmpeg.mp4```
    - For NIDAQ concatenation:
        - - ```conda activate daq```
        - ```ffmpeg -f concat -safe 0 -i {path_to_text_file} -c copy {path_to_output_file}```
            
            eg: ```ffmpeg -f concat -safe 0 -i D:\big_setup\experiment_9\concatenated_data_cam_mic_sync\channel_0.txt -c copy D:\big_setup\experiment_9\concatenated_data_cam_mic_sync\channel_0_ffmpeg.wav```

In [18]:
no_audio_files = int(len(glob.glob(path_concat + "*channel_*.txt"))/no_channels)
no_video_files = int(len(glob.glob(path_concat + "*video_*.txt"))/len(camera_names))

In [19]:
if no_audio_files != no_video_files:
    raise Exception("Number of audio and video files don't match")
else:
    matched_csv = []
    f = open(path_final+"README.txt", 'w')
    f.write("The files that are synced are as follows:\n")
    f.write("Video file names - corresponding audio file names \n")
    for i in range(no_audio_files):
        temp_0 = []
        temp_1 = []
        for j in range(no_channels):
            temp_0.append(f"channel_{j}_{i}")
        for j in camera_names:
            temp_1.append(f"video_{j}_{i}")
        f.write(f"[{', '.join(temp_1)}] - [{', '.join(temp_0)}]\n")
        matched_csv.append([temp_1,temp_0])
    pd.DataFrame(matched_csv,columns=["video","audio"]).to_csv(path_final+"sync.csv", index = False)
    f.close()

In [20]:
filename_txt = path_concat + "*.txt"
txt_files = glob.glob(filename_txt)
video_files = []
audio_files = []

for file in txt_files:
    name = file.split('\\')[-1].split(".")[0]
    full_path_txt = path_concat+name+".txt"
    full_path_txt = full_path_txt.replace("/","\\")
    if name[0] == "c":
        full_path_wav = path_final+name+".wav"
        full_path_wav = full_path_wav.replace("/","\\")
        !ffmpeg -f concat -safe 0 -i {full_path_txt} -c copy  {full_path_wav}
        # -rf64 auto
    else:
        full_path_mp4 = path_final+name+".mp4"
        full_path_mp4 = full_path_mp4.replace("/","\\")
        !ffmpeg -f concat -safe 0 -i {full_path_txt} -c copy {full_path_mp4}

ffmpeg version 4.3.1 Copyright (c) 2000-2020 the FFmpeg developers
  built with gcc 10.2.1 (GCC) 20200726
  configuration: --disable-static --enable-shared --enable-gpl --enable-version3 --enable-sdl2 --enable-fontconfig --enable-gnutls --enable-iconv --enable-libass --enable-libdav1d --enable-libbluray --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libopus --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libsrt --enable-libtheora --enable-libtwolame --enable-libvpx --enable-libwavpack --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libzimg --enable-lzma --enable-zlib --enable-gmp --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvo-amrwbenc --enable-libmysofa --enable-libspeex --enable-libxvid --enable-libaom --enable-libgsm --enable-librav1e --disable-w32threads --enable-libmfx --enable-ffnvcodec --enable-cuda-llvm --enable-cuvid --enable-d3d11va 